# Imports


In [12]:
# ------------------------------
# 1. IMPORTS
# ------------------------------
print("Stage: 1 - Imports completed")

from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
# import torch
import textstat

from typing import List, Dict, Optional, Tuple
from owlready2 import get_ontology, Thing
# import math

Stage: 1 - Imports completed


Ontology loading and adaptive metrics

In [13]:
############################################
# ONTOLOGY LOADING + ADAPTIVE METRICS
############################################

def load_ontology(owl_path: str):
    print(f"Stage: LOADING ONTOLOGY from '{owl_path}'")
    ont = get_ontology(owl_path).load()
    print("Stage: ONTOLOGY loaded")
    
    # PRE-COMPUTE GLOBAL STATS FOR NORMALIZATION
    # This prevents recalculating max_depth/total_classes for every entity
    global MAX_DEPTH, TOTAL_CLASSES, OBJECT_PROPERTIES
    classes = list(ont.classes())
    TOTAL_CLASSES = len(classes)
    OBJECT_PROPERTIES = list(ont.object_properties())
    
    # Estimate Max Depth (Heuristic scan)
    # print("Stage: Pre-computing ontology stats...")
    current_max = 0
    for c in classes[:500]: # Sample first 500 for speed, or remove slice for accuracy
        d = _class_depth_raw(c)
        if d > current_max: current_max = d
    MAX_DEPTH = current_max if current_max > 0 else 12 # Default fallback
    
    # print(f"Stage: Stats - Total Classes: {TOTAL_CLASSES}, Est. Max Depth: {MAX_DEPTH}")
    return ont

def find_concept(ontology, label_or_name: str):
    # (Same as your original code)
    # print(f"Stage: FIND_CONCEPT searching for '{label_or_name}'")
    candidates = list(ontology.search(label=label_or_name))
    if not candidates:
        candidates = [
            c for c in ontology.classes() 
            if any(str(lbl).lower() == label_or_name.lower() 
                   for lbl in getattr(c, 'label', []))
        ]
    if candidates:
        return candidates[0]
    try:
        return ontology[label_or_name]
    except Exception:
        return None

def _class_depth_raw(concept) -> int:
    # Helper without print statements for internal calculations
    try:
        ancestors = [a for a in concept.ancestors() if isinstance(a, type)]
        if Thing in ancestors:
            ancestors = [a for a in ancestors if a is not Thing]
        return max(len(ancestors), 0)
    except:
        return 0

def _class_depth(concept) -> int:
    # Wrapper for external calls
    return _class_depth_raw(concept)

# ---------------------------------------------------------
# NEW: PAPER-BASED METRICS (Venugopal & Kumar, 2020)
# ---------------------------------------------------------

def calculate_suitability_score(concept, user_category: str) -> float:
    """
    Calculates a suitability score for an ancestor based on User Category.
    Uses Specificity (Normalized Depth) and Popularity (Normalized Usage).
    """
    # 1. Calculate Specificity (S) 
    raw_depth = _class_depth_raw(concept)
    specificity = raw_depth / MAX_DEPTH if MAX_DEPTH > 0 else 0
    
    # 2. Calculate Popularity (P) 
    # "number of object properties which are linked to it"
    incoming_links = 0
    for prop in OBJECT_PROPERTIES:
        # Check if this concept is the target (Range) of a property
        if concept in prop.range:
            incoming_links += 1
            
    popularity = incoming_links / TOTAL_CLASSES if TOTAL_CLASSES > 0 else 0
    
    # 3. User-Adaptive Scoring Logic
    cat = user_category.upper()
    score = 0.0
    
    if cat == 'BEGINNER':
        # Beginners need High Popularity (Familiarity) and Low Specificity (Generality)
        # Weights: 60% Popularity, 40% Generality
        score = (0.6 * popularity) + (0.4 * (1.0 - specificity))
        
    elif cat == 'EXPERT':
        # Experts need High Specificity (Technical Depth)
        # Popularity is irrelevant for experts (they know obscure terms)
        score = specificity
        
    elif cat == 'INTERMEDIATE':
        # Intermediates need a balance. We target the "middle" of the taxonomy.
        # This creates a bell curve preference around 0.5 specificity.
        score = 1.0 - abs(specificity - 0.5)
        
    return score

def get_ancestors(concept, include_self=False):
    ancestors = [a for a in concept.ancestors() if isinstance(a, type) and a is not Thing]
    # Sort by depth for display purposes
    ancestors_sorted = sorted(ancestors, key=lambda x: _class_depth_raw(x))
    return [concept] + ancestors_sorted if include_self else ancestors_sorted

def select_ancestors(entity_concept, user_category):
    """
    Selects top-k ancestors based on the Adaptive Suitability Score.
    Note: ic_thresholds is deprecated in this new version but kept for signature compatibility.
    """
    # print(f"Stage: SELECT_ANCESTORS for '{getattr(entity_concept,'name',str(entity_concept))}' (User: {user_category})")
    
    if entity_concept is None:
        return []

    # 1. Get all candidates
    ancestors = get_ancestors(entity_concept)
    
    # 2. Score candidates
    scored_candidates = []
    for anc in ancestors:
        score = calculate_suitability_score(anc, user_category)
        scored_candidates.append((anc, score))
        
    # 3. Sort by Score (Descending) - Best fit first
    scored_candidates.sort(key=lambda x: x[1], reverse=True)
    # print("** Number of candidates found:", len(scored_candidates))
    
    # 4. Selection Heuristics
    
    if user_category.upper() == 'EXPERT':
        # Experts can handle more context if it's highly specific
        # top_k = max(1, math.ceil(len(scored_candidates) * 0.8))  # All
        top_k = 4
    elif user_category.upper() == 'BEGINNER':
        # Beginners get fewer, more general concepts
        # top_k = max(1, math.ceil(len(scored_candidates) * 0.4))  # Top 40%
        top_k = 3
    elif user_category.upper() == 'INTERMEDIATE':
        # Intermediates get a moderate amount
        # top_k = max(1, math.ceil(len(scored_candidates) * 0.6))  # Top 60%
        top_k = 3
    
    selected_tuples = scored_candidates[:top_k]
    
    # 5. Extract Labels
    def label(c):
        return str(getattr(c, 'label', [c.name])[0])

    labels = [label(a) for a, _ in selected_tuples]
    
    # Re-sort the selected ones by natural hierarchy (Depth) for readability in the final text
    # (We selected the *best* ones, now we order them logically)
    # Mapping back to objects to sort by depth
    selected_objs = [obj for obj, score in selected_tuples]
    selected_objs.sort(key=lambda x: _class_depth_raw(x))
    final_labels = [label(o) for o in selected_objs]

    # print(f"Stage: Selected: {final_labels}")
    return final_labels

# Task at hand and integrate ontology

In [14]:
############################################
# 2. MEDICAL NER MODEL
############################################

def load_medical_ner_model():
    print("Stage: LOADING MEDICAL NER MODEL")
    model_name = "d4data/biomedical-ner-all"
    # If you want BioBERT NER: d4data/biomedical-ner-all
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForTokenClassification.from_pretrained(model_name)
    nlp = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")
    print("Stage: MEDICAL NER MODEL loaded\n")
    return nlp


############################################
# 3. INTEGRATE ONTOLOGY WITH NER OUTPUT
############################################

def explain_entities_with_ontology(text, ner_model, ontology, user_category):
    # print("Stage: EXPLAIN_ENTITIES_WITH_ONTOLOGY - start")
    # print(f"Stage: running NER on text of length {len(text)}")
    ner_results = ner_model(text)
    # print(f"Stage: NER returned {len(ner_results)} entities")

    explanations = []

    _allowed_tags = {
        "Sign_symptom",
        "Clinical_event",
        "Diagnostic_procedure",
        "Therapeutic_procedure",
        "Disease_disorder",
    }

    for idx, ent in enumerate(ner_results, start=1):

        if ent.get('entity_group') not in _allowed_tags:
            # print(f"Stage: Skipping entity '{ent['word']}' with tag '{ent.get('entity_group')}'")
            continue

        entity_text = ent['word']
        # print(f"\nStage: Processing entity {idx}: '{entity_text}' (label: {ent.get('entity_group')})")

        # Map to ontology
        concept = find_concept(ontology, entity_text)

        if concept is None:
            # print(f"Stage: No ontology match for '{entity_text}'")
            explanations.append({
                'entity': entity_text,
                'concept_found': False,
                'explanation': "No ontology match found."
            })
            continue

        # print(f"Stage: Ontology concept '{concept.name}' found for entity '{entity_text}'")

        # Select ancestors based on user level
        selected = select_ancestors(
            entity_concept=concept,
            user_category=user_category
        )

        explanations.append({
            'entity': entity_text,
            'ontology_class': ent.get('entity_group'),
            'concept_found': True,
            'ancestors_selected': selected
        })

    # print("Stage: EXPLAIN_ENTITIES_WITH_ONTOLOGY - completed")
    return explanations


# LLM for Natural Language Explanation

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device_ = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [5]:
# def generate_constrained_explanation(entity, entity_class, ancestors, user_category):
#     # 1. Create the Semantic Skeleton (Triples)
#     triples = []
    
#     # We simplify the triples for the prompt to reduce "noise"
#     if user_category == "BEGINNER":
#         triples.append(f"({entity} -> is identified as -> {entity_class})")
#         for anc in ancestors:
#             triples.append(f"({entity} -> is a type of -> {anc})")
            
#     elif user_category == "EXPERT":
#         triples.append(f"({entity} -> classified as -> {entity_class})")
#         for anc in ancestors:
#             triples.append(f"({entity} -> subclass of -> {anc})")
    
#     else: # INTERMEDIATE
#         triples.append(f"({entity} -> falls under category -> {entity_class})")
#         for anc in ancestors:
#             triples.append(f"({entity} -> is a type of -> {anc})")

#     # 2. The Constrained Prompt
#     # CHANGES:
#     # - Removed "Justify" (Trigger word)
#     # - Added "Pattern" (Positive constraint)
#     # - Added "Forbidden" list (Negative constraint reinforcement)
    
#     prompt = f"""
# ### TASK: 
# Act as a Data-to-Text Converter. Convert the provided "DATA TRIPLES" into a single natural sentence. 

# ### INPUT DATA:
# Target Audience: {user_category}
# DATA TRIPLES:
# {chr(10).join(triples)}

# ### STRICT CONSTRAINTS:
# 1. **NO EXTERNAL KNOWLEDGE**: Do not explain *what* the disease is. Do not mention symptoms, biology, blood sugar, nerves, or body parts unless they are explicitly written in the DATA TRIPLES.
# 2. **VERBALIZE ONLY**: Your job is only to turn the arrows (->) into words.
# 3. **PATTERN**: Start the sentence with "{entity} is identified as {entity_class}..." and connect the ancestors using "because it is...".

# ### RESPONSE:
# """
    
#     return prompt

In [ ]:
def generate_constrained_explanation(entity, ancestors, user_category):
    # 1. Create the Semantic Skeleton (Triples)
    triples = []
    if user_category == "BEGINNER":
        # Beginners get "is a type of" relations
        for anc in ancestors:
            triples.append(f"({entity} -> is a type of -> {anc})")
            
    elif user_category == "EXPERT":
        # Experts get specific taxonomic relations
        for anc in ancestors:
            triples.append(f"({entity} -> subclass of -> {anc})")
        
    else: # INTERMEDIATE
        # Intermediates get a mix
        for anc in ancestors:
            triples.append(f"({entity} -> is a type of -> {anc})")

    # 2. The Constrained Prompt (RDF-to-Text)
    prompt = f"""
    TASK: Convert the following structured data into a natural sentence.
    TARGET AUDIENCE: {user_category}
    
    DATA TRIPLES:
    {chr(10).join(triples)}
    
    RULES:
    1. ONLY use the information in the Data Triples.
    2. Do NOT add definitions, symptoms, or outside facts.
    3. Make it grammatically correct.

    RESPONSE:
    """
    
    return prompt, len(prompt)

In [52]:
def generate_reasoning(entity, ancestors, context, entity_class, user_categor, max_new_tokens=200):

    prompt, prompt_length = generate_constrained_explanation(entity,entity_class ,ancestors, user_categor)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens
    )

    reasoning = tokenizer.decode(output[0], skip_special_tokens=True)
    return reasoning.split('RESPONSE:\n')[-1].strip()


# Testing with 1 text

In [67]:
# text = "The patient was diagnosed with type 2 diabetes mellitus and hypertension. He also showed early signs of diabetic neuropathy and mild renal impairment. Metformin was prescribed along with regular blood sugar monitoring and lifestyle modifications to manage his chronic conditions."
text = "Results of investigation into the new-onset cardiomyopathy included normal cardiac enzyme levels, an electrocardiogram (ECG) that revealed no ischemic changes, and a coronary angiogram of normal appearance."
ontology = load_ontology("doid.owl")
ner_model = load_medical_ner_model()

user_category = "EXPERT"  # Options: BEGINNER, INTERMEDIATE, EXPERT


Stage: LOADING ONTOLOGY from 'doid.owl'
Stage: ONTOLOGY loaded
Stage: LOADING MEDICAL NER MODEL


Device set to use cpu


Stage: MEDICAL NER MODEL loaded



In [71]:

output = explain_entities_with_ontology(
    text=text,
    ner_model=ner_model,
    ontology=ontology,
    user_category=user_category
)

In [72]:
entity_info = []
for item in output:
    if item['concept_found']:
        if item['entity'] in item['ancestors_selected']:
            item['ancestors_selected'].remove(item['entity'])
        entity_info.append(item)
        print(item)

{'entity': 'cardiomyopathy', 'ontology_class': 'Disease_disorder', 'concept_found': True, 'ancestors_selected': ['cardiovascular system disease', 'thoracic disease', 'heart disease']}


In [73]:
for i in entity_info:
    entity = i['entity']
    ancestors = i['ancestors_selected']
    context = text
    entity_class = i['ontology_class']

    reasoning = generate_reasoning(entity, ancestors, context, entity_class, user_category).strip()
    print("\n--------------------------------------------------------------------------------------\n")
    print("Entity:", entity)
    print("Reasoning:")
    print(reasoning.split('Output Sentence:')[-1].strip())
    print("Readability Scores of Generated Explanation:")

    readability_scores = {
        "Flesch Reading Ease": textstat.flesch_reading_ease(reasoning),
        "Flesch-Kincaid Grade Level": textstat.flesch_kincaid_grade(reasoning)
        }

    for k, v in readability_scores.items():
        print(f"{k}: {v}")



--------------------------------------------------------------------------------------

Entity: cardiomyopathy
Reasoning:
Cardiomyopathy is classified as a Disease_disorder, specifically characterized as a cardiovascular system disease and thoracic disease, and also categorized as a heart disease.
Readability Scores of Generated Explanation:
Flesch Reading Ease: -3.153181818181821
Flesch-Kincaid Grade Level: 19.164545454545458


In [74]:
r = reasoning.split('Output Sentence:')[-1].strip()

readability_scores = {
    "Flesch Reading Ease": textstat.flesch_reading_ease(r),
    "Flesch-Kincaid Grade Level": textstat.flesch_kincaid_grade(r)
    }

for k, v in readability_scores.items():
    print(f"{k}: {v}")

Flesch Reading Ease: -7.77956521739128
Flesch-Kincaid Grade Level: 20.05826086956522


In [ ]:
r = reasoning.split('Output Sentence:')[-1].strip()
print(r)

readability_scores = {
    "Flesch Reading Ease": textstat.flesch_reading_ease(r),
    "Flesch-Kincaid Grade Level": textstat.flesch_kincaid_grade(r)
    }

for k, v in readability_scores.items():
    print(f"{k}: {v}")

Cardiomyopathy is identified as a Disease_disorder because it is a disease affecting the cardiovascular system.
Flesch Reading Ease: -0.14999999999997726
Flesch-Kincaid Grade Level: 17.006666666666664


----

In [19]:
print(reasoning.split(':\n')[-1].strip())

> Diabetic neuropathy is a type of disease that affects the nervous system.


# Automating some texts for getting average FRE and FKGL

In [9]:
import os
import glob
import pandas as pd
import textstat
from tqdm import tqdm  # For a progress bar

In [14]:
def process_maccrobat_dataset(
    directory_path: str, 
    target_count: int = 5, 
    user_category: str = "BEGINNER"
):
    """
    Iterates through .txt files in the directory, extracts entities, 
    generates explanations, and saves metrics to CSV.
    """
    
    # 1. Get all txt files
    file_paths = glob.glob(os.path.join(directory_path, "*.txt"))
    print(f"Found {len(file_paths)} files in {directory_path}")
    
    collected_data = []
    explanation_count = 0
    
    # 2. Iterate through files
    # Using tqdm for a progress bar
    pbar = tqdm(total=target_count, desc="Generating Explanations")
    
    for file_path in file_paths:
        # print(f"\nProcessing file: {file_path}")
        if explanation_count >= target_count:
            break
            
        filename = os.path.basename(file_path)
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except Exception as e:
            print(f"Error reading {filename}: {e}")
            continue
        count = 0
        # 3. Iterate through lines in the file
        for line in lines:
            count += 1
            # print(f"Processing line {count}\n")
            if explanation_count >= target_count:
                break
                
            line = line.strip()
            if not line or len(line) < 20: # Skip empty or very short lines
                continue
                
            # A. Run your existing pipeline on this line
            # This calls NER -> Ontology Mapping -> Ancestor Selection
            try:
                entity_results = explain_entities_with_ontology(
                    text=line,
                    ner_model=ner_model,
                    ontology=ontology,
                    user_category=user_category
                )
            except Exception as e:
                print(f"Pipeline error on line: {e}")
                continue
            
            # B. Process valid entities found
            for item in entity_results:
                if explanation_count >= target_count:
                    break
                
                # We only care if an ontology concept was found
                if item.get('concept_found') is True:
                    
                    entity_text = item['entity']
                    ancestors = item['ancestors_selected']
                    entity_class = item['ontology_class']
                    
                    # Safety check: Remove entity from ancestors if it appears there
                    if entity_text in ancestors:
                        ancestors.remove(entity_text)
                    
                    # C. Generate Natural Language Explanation (Gemma/LLM)
                    try:
                        print("Generating Explanations")
                        # calling your existing function
                        explanation = generate_reasoning(
                            entity=entity_text,
                            ancestors=ancestors,
                            context=line,
                            entity_class=entity_class,
                            user_categor=user_category # Matching your variable name
                        ).strip()
                        
                        # Clean up artifacts if the LLM output keeps the prompt headers
                        if "RESPONSE:" in explanation:
                            explanation = explanation.split("RESPONSE:")[-1].strip()
                        if ':\n' in explanation:
                            explanation = explanation.split(':\n')[-1].strip()
                        
                    except Exception as e:
                        print(f"Generation error: {e}")
                        explanation = "Error in generation"
                    
                    # D. Calculate Readability Scores
                    fre = textstat.flesch_reading_ease(explanation)
                    fkgl = textstat.flesch_kincaid_grade(explanation)
                    
                    # E. Store Data
                    row = {
                        "File": filename,
                        "User_Category": user_category,
                        "Original_Sentence": line,
                        "Entity": entity_text,
                        "Ontology_Class": entity_class,
                        "Selected_Ancestors": ", ".join(ancestors),
                        "Generated_Explanation": explanation,
                        "FRE_Score": fre,
                        "FKGL_Score": fkgl
                    }
                    
                    collected_data.append(row)
                    explanation_count += 1
                    pbar.update(1)

    pbar.close()
    
    # 4. Save to CSV
    df = pd.DataFrame(collected_data)
    output_csv = f"maccrobat_results_{user_category.lower()}.csv"
    df.to_csv(output_csv, index=False)
    
    print(f"\nProcessing Complete.")
    print(f"Generated {len(df)} explanations.")
    print(f"Saved to {output_csv}")
    
    # # Display average stats
    # if not df.empty:
    #     print("\nAverage Metrics:")
    #     print(f"Avg FRE: {df['FRE_Score'].mean():.2f}")
    #     print(f"Avg FKGL: {df['FKGL_Score'].mean():.2f}")
        
    return df


In [15]:
# ==========================================
# EXECUTE THE FUNCTION
# ==========================================

# Make sure MACCROBAT2020 folder exists in your current directory
if not os.path.exists("MACCROBAT2020"):
    print("Error: Directory 'MACCROBAT2020' not found. Please create it and upload txt files.")
else:
    # Run for BEGINNER
    df_results = process_maccrobat_dataset(
        directory_path="MACCROBAT2020",
        target_count=50,
        user_category="EXPERT"
    )

    # Optional: Preview the data
    print("\nFirst 5 rows:")
    display(df_results.head())

Found 200 files in MACCROBAT2020


Generating Explanations:   0%|          | 0/50 [00:00<?, ?it/s]

Generating Explanations


Generating Explanations:   2%|▏         | 1/50 [00:25<20:44, 25.39s/it]

Generating Explanations


...

Generating Explanations:  98%|█████████▊| 49/50 [12:34<00:14, 14.70s/it]

Generating Explanations


Generating Explanations: 100%|██████████| 50/50 [12:52<00:00, 15.45s/it]


Processing Complete.
Generated 50 explanations.
Saved to maccrobat_results_expert.csv

First 5 rows:


,File,User_Category,Original_Sentence,Entity,Ontology_Class,Selected_Ancestors,Generated_Explanation,FRE_Score,FKGL_Score
0,15939911.txt,EXPERT,Contrast echocardiography using saline reveale...,patent foramen ovale,Disease_disorder,"congenital heart disease, heart septal defect,...",The patent foramen ovale is a subclass of cong...,55.829454,10.523607
1,16778410.txt,EXPERT,The patient was a 34-yr-old man who presented ...,fever,Sign_symptom,"symptom, neurological and physiological symptom",The data triples provide insights into the sub...,15.403529,15.334118
2,16778410.txt,EXPERT,He was a smoker and had a history of pulmonary...,tuberculosis,Disease_disorder,"disease by infectious agent, bacterial infecti...",The data triples provide a classification hier...,17.932308,13.987692
3,17803823.txt,EXPERT,A 23 year old white male with a 4 year history...,vomiting,Sign_symptom,"symptom, digestive system symptom",The data triples provide insights into the sub...,22.412000,16.344000
4,17803823.txt,EXPERT,"There was also a rash on the elbows, ankles an...",rash,Sign_symptom,"symptom, skin and integumentary tissue symptom",The rash is a subclass of the symptom of skin ...,69.993846,6.726154


In [17]:
dfre_avg = df_results['FRE_Score'].mean()
fkgl_avg = df_results['FKGL_Score'].mean()

print(f"\nOverall Average Flesch Reading Ease (FRE): {dfre_avg:.2f}")
print(f"Overall Average Flesch-Kincaid Grade Level (FKGL): {fkgl_avg:.2f}")


Overall Average Flesch Reading Ease (FRE): 36.34
Overall Average Flesch-Kincaid Grade Level (FKGL): 12.42


In [18]:
df_easy = pd.read_csv("maccrobat_results_beginner.csv")
dfeasy_avg = df_easy['FRE_Score'].mean()
fkgleasy_avg = df_easy['FKGL_Score'].mean()

print(f"Beginner Average Flesch Reading Ease (FRE): {dfeasy_avg:.2f}")
print(f"Beginner Average Flesch-Kincaid Grade Level (FKGL): {fkgleasy_avg:.2f}")

Beginner Average Flesch Reading Ease (FRE): 52.15
Beginner Average Flesch-Kincaid Grade Level (FKGL): 9.40


---